# Fold runner for phase 10.
# We just copied everything from phase 6 and ran that for fold1, fold2, fold3

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ndcg_score
import lightgbm as lgb
import json
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT=Path.cwd().parent
DATA_RAW=PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED=PROJECT_ROOT / 'data' / 'processed'
PHASE5_OUTPUT=DATA_PROCESSED / 'phase5_failure_analysis'
PHASE6_FOLDS_OUTPUT=DATA_PROCESSED / 'phase6_models_folds'
PHASE6_FOLDS_OUTPUT.mkdir(parents=True, exist_ok=True)



PIPELINES=['raw', 'global', 'per_query']
K_VALUES=[1, 3, 5, 10]
RANDOM_SEED=36
MAX_PAIRS_PER_QUERY=1000

FOLD_NUMS=[1, 2, 3]
MODELS=['pointwise','pairwise','lightgbm']
EXPECTED_CONFIGS=len(MODELS) * len(PIPELINES)

print(f"Phase 6 fold output: {PHASE6_FOLDS_OUTPUT}")
print(f"Folds: {FOLD_NUMS}, Expected configs/fold: {EXPECTED_CONFIGS}")

Phase 6 fold output: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase6_models_folds
Folds: [1, 2, 3], Expected configs/fold: 9


## 1. Data Loading

In [2]:
print("="*80)
print("DATA LOADING")
print("="*80)

def load_letor_fold(dataset_name, fold_num):
    base_path=DATA_RAW / dataset_name / f'Fold{fold_num}'
    splits={}
    
    for split_name in ['train', 'vali', 'test']:
        file_path=base_path / f'{split_name}.txt'
        if not file_path.exists():
            raise RuntimeError(f"Missing: {file_path}")
        
        data=[]
        with open(file_path, 'r') as f:
            for line in f:
                if '#' in line:
                    line=line[:line.index('#')]
                parts=line.strip().split()
                if len(parts)<2:
                    continue
                
                label=int(parts[0])
                qid=int(parts[1].split(':')[1])
                features={}
                for item in parts[2:]:
                    if ':' in item:
                        fid, fval=item.split(':')
                        fid=int(fid)
                        if 1<=fid<=46:
                            features[f'f{fid}']=float(fval)
                row={'qid': qid, 'label': label}
                row.update(features)
                data.append(row)
        
        df=pd.DataFrame(data)
        for i in range(1, 47):
            if f'f{i}' not in df.columns:
                df[f'f{i}']=0.0
        df=df.drop(columns=[f'f{i}' for i in range(6, 11)], errors='ignore')
        splits[split_name]=df
        print(f"{dataset_name} {split_name:5s}: {len(df):6d} docs, {df['qid'].nunique():4d} queries")
    
    return splits['train'], splits['vali'], splits['test']

print("\nMQ2007:")
train_2007, vali_2007, test_2007=load_letor_fold('MQ2007', 1)
print("\nMQ2008:")
_, _, test_2008=load_letor_fold('MQ2008', 1)

feature_cols=[c for c in train_2007.columns if c.startswith('f')]
print(f"\nFeatures: {len(feature_cols)}")
print("DATA LOADED")

DATA LOADING

MQ2007:
MQ2007 train:  42158 docs, 1017 queries
MQ2007 vali :  13813 docs,  339 queries
MQ2007 test :  13652 docs,  336 queries

MQ2008:
MQ2008 train:   9630 docs,  471 queries
MQ2008 vali :   2707 docs,  157 queries
MQ2008 test :   2874 docs,  156 queries

Features: 41
DATA LOADED


## 2. Metrics

In [3]:
def precision_at_k(ranked_labels, k, relevance_threshold=1):
    if k<=0:
        raise ValueError("k must be positive")
    if relevance_threshold not in [1, 2]:
        raise ValueError("relevance_threshold must be 1 or 2")
    k=min(k, len(ranked_labels))
    if k==0:
        return 0.0
    top_k=ranked_labels[:k]
    relevant=sum(1 for label in top_k if label >= relevance_threshold)
    return relevant / k

def ndcg_at_k(ranked_labels, k):
    if k<=0:
        raise ValueError("k must be positive")
    k=min(k, len(ranked_labels))
    if k==0:
        return 0.0
    
    def dcg(labels, k):
        labels_k=labels[:k]
        gains=[2**label - 1 for label in labels_k]
        discounts=[np.log2(i + 2) for i in range(len(labels_k))]
        return sum(g / d for g, d in zip(gains, discounts))
    
    dcg_val=dcg(ranked_labels, k)
    ideal=sorted(ranked_labels, reverse=True)
    idcg_val=dcg(ideal, k)
    return dcg_val / idcg_val if idcg_val > 0 else 0.0

def failure_at_k(ranked_labels, k, relevance_threshold=1):
    """Empty query returns 0.0 (not evaluable)."""
    if k<=0:
        raise ValueError("k must be positive")
    if relevance_threshold not in [1, 2]:
        raise ValueError("relevance_threshold must be 1 or 2")
    k=min(k, len(ranked_labels))
    if k==0:
        return 0.0  
    top_k=ranked_labels[:k]
    has_relevant=any(label >= relevance_threshold for label in top_k)
    return 0.0 if has_relevant else 1.0

print("Metrics defined")

Metrics defined


## 3. NDCG Verification

In [4]:
print("="*80)
print("NDCG VERIFICATION ")
print("="*80)

sample_qids=test_2007['qid'].unique()[:5]
print(f"\n{'QID':<8s} {'Custom':<10s} {'sklearn':<10s} {'Diff':<12s}")
print("-"*45)

for qid in sample_qids:
    query_docs=test_2007[test_2007['qid']==qid].copy()
    n_docs=len(query_docs)
    
    if n_docs==0:
        continue
    
    #Creating deterministic scores in original order
    query_docs=query_docs.reset_index(drop=True)
    tmp_scores=np.linspace(1.0, 0.0, n_docs)
    query_docs['tmp_score']=tmp_scores
    
    k=min(5, n_docs)
    if k==0:
        continue
    
    #Custom NDCG: ranking by score, computing on ranked labels
    sorted_docs=query_docs.sort_values('tmp_score', ascending=False, kind='stable')
    ranked_labels=sorted_docs['label'].values
    custom=ndcg_at_k(ranked_labels, k)
    
    #sklearn NDCG: using original order 
    original_labels=query_docs['label'].values
    original_scores=query_docs['tmp_score'].values
    y_true_gains=np.array([[2**label - 1 for label in original_labels]])
    y_score=original_scores.reshape(1, -1)
    sklearn_ndcg=ndcg_score(y_true_gains, y_score, k=k)
    
    diff=abs(custom - sklearn_ndcg)
    print(f"{qid:<8d} {custom:<10.6f} {sklearn_ndcg:<10.6f} {diff:<12.9f}")
    
    if diff>1e-6:
        print(f"WARNING: diff={diff:.2e}")

print("\n NDCG verified ")
print("="*80)

NDCG VERIFICATION 

QID      Custom     sklearn    Diff        
---------------------------------------------
7968     0.214533   0.214533   0.000000000 
7979     0.000000   0.000000   0.000000000 
7993     0.131205   0.131205   0.000000000 
7995     0.195190   0.195190   0.000000000 
8002     0.277273   0.277273   0.000000000 

 NDCG verified 


## 4. Loading Phase 5 Threshold

In [5]:
print("\n"+"="*80)
print("LOADING PHASE 5 THRESHOLD")
print("="*80)

threshold_file=PHASE5_OUTPUT / 'threshold_sweep_sensitivity.csv'
if not threshold_file.exists():
    raise RuntimeError(f"Missing: {threshold_file}")

threshold_df=pd.read_csv(threshold_file)
threshold_row=threshold_df[threshold_df['percentile'] == 25]
if len(threshold_row)==0:
    raise RuntimeError("25th percentile not found")

FLAT_SCORE_THRESHOLD=threshold_row.iloc[0]['threshold']
#Pandas still return a dataframe even if it is only one row, so we need iloc[0]
print(f"\nThreshold: {FLAT_SCORE_THRESHOLD:.4f} (from Phase 5, NOT recomputed)")
print("="*80)


LOADING PHASE 5 THRESHOLD

Threshold: 0.0414 (from Phase 5, NOT recomputed)


## 5. Query level evaluation

In [6]:
def evaluate_query_level(predictions_df, k_values=[1, 3, 5, 10]):
    results=[]      #collect one dictionary per query
    for qid, group in predictions_df.groupby('qid'):
        group_sorted=group.sort_values('score', ascending=False, kind='stable').reset_index(drop=True)
        ranked_labels=group_sorted['label'].values
        ranked_scores=group_sorted['score'].values
        n_docs=len(ranked_labels)
        
        query_metrics={
            'qid': qid,
            'num_docs': n_docs,
            'num_relevant_1': (group['label'] >= 1).sum(),
            'num_relevant_2': (group['label'] == 2).sum()
        }
        
        for k in k_values:
            query_metrics[f'P@{k}_primary']=precision_at_k(ranked_labels, k, 1)
            query_metrics[f'P@{k}_sensitivity']=precision_at_k(ranked_labels, k, 2)
            query_metrics[f'NDCG@{k}']=ndcg_at_k(ranked_labels, k)
            query_metrics[f'Failure@{k}_primary']=failure_at_k(ranked_labels, k, 1)
            query_metrics[f'Failure@{k}_sensitivity']=failure_at_k(ranked_labels, k, 2)
            
            #Using k_actual for tie-at-cutoff
            k_actual=min(k, n_docs)
            top_k=ranked_scores[:k_actual]
            query_metrics[f'unique_scores@{k}']=len(np.unique(top_k))
            
            #Tie-at-cutoff only when k_actual < n_docs
            if k_actual < n_docs:
                query_metrics[f'tie_at_cutoff@{k}']=(ranked_scores[k_actual-1]==ranked_scores[k_actual])
            else:
                query_metrics[f'tie_at_cutoff@{k}']=False
        
        top10=ranked_scores[:min(10, n_docs)]
        query_metrics['score_range_top10']=top10.max() - top10.min() if len(top10) > 0 else 0.0
        query_metrics['score_std_top10']=top10.std() if len(top10) > 1 else 0.0
        results.append(query_metrics)
    return pd.DataFrame(results)

print("Evaluation defined")

Evaluation defined


## 6. Normalization

In [7]:
def apply_normalization(train_df, test_df, pipeline_name, feature_cols, fitted_scaler=None):
    #Fitting scaler on train only, transforming test with fitted scaler.
    train_norm=train_df.copy()
    test_norm=test_df.copy()
    scaler=fitted_scaler
    
    if pipeline_name=='raw':
        pass
    elif pipeline_name=='global':
        if scaler is None:
            scaler=StandardScaler()
            train_norm[feature_cols]=scaler.fit_transform(train_df[feature_cols])
        else:
            train_norm[feature_cols]=scaler.transform(train_df[feature_cols])
        test_norm[feature_cols]=scaler.transform(test_df[feature_cols])
    elif pipeline_name=='per_query':
        for df in [train_norm, test_norm]:
            for qid, idx in df.groupby('qid').groups.items():
                feats=df.loc[idx, feature_cols]
                means=feats.mean()
                stds=feats.std(ddof=0).replace(0, 1) #preventing division by 0
                df.loc[idx, feature_cols]=(feats - means) / stds
    else:
        raise ValueError(f"Unknown: {pipeline_name}")
    
    return train_norm, test_norm, scaler

print("Normalization defined")

Normalization defined


## 7. Pairwise Ranker

In [8]:
class PairwiseLogisticRanker:
    def __init__(self, random_state=36, max_pairs_per_query=1000):
        self.model=LogisticRegression(max_iter=1000, random_state=random_state, solver='lbfgs', class_weight='balanced')
        self.feature_cols=None
        self.max_pairs=max_pairs_per_query
        self.random_state=random_state
    
    def _generate_pairs(self, df, feature_cols):
        pairs_X, pairs_y= [], []
        #pairs_X=feature differences between two docs
        #pairs_y=1 if first doc should rank higher, else 0
        rng=np.random.RandomState(self.random_state)
        for qid, group in df.groupby('qid'):
            docs=group.reset_index(drop=True)
            n=len(docs)
            all_pairs=[]
            for i in range(n):
                for j in range(i+1, n):
                    #This generates each unique pair once
                    #if n=4, pairs are (0,1), (0,2), (0,3), (1,2), (1,3), (2,3)
                    #Only keeping pairs with different labels as if labels are equal, the pair gives no "which is better" signal
                    li, lj=docs.loc[i, 'label'], docs.loc[j, 'label']
                    if li!=lj:
                        all_pairs.append((i, j, li, lj))
            if len(all_pairs)>self.max_pairs:
                indices=rng.choice(len(all_pairs), self.max_pairs, replace=False)
                all_pairs=[all_pairs[idx] for idx in indices]
            for i, j, li, lj in all_pairs:
                diff=docs.loc[i, feature_cols].values - docs.loc[j, feature_cols].values
                pairs_X.append(diff)
                pairs_y.append(1 if li > lj else 0)
        pairs_X=np.array(pairs_X)
        pairs_y=np.array(pairs_y)
        if len(np.unique(pairs_y)) < 2:
            raise RuntimeError("Only one class")
        return pairs_X, pairs_y
    
    def fit(self, train_df, feature_cols):
        self.feature_cols= feature_cols
        print(f"Generating pairs (max {self.max_pairs}/query)...")
        X, y=self._generate_pairs(train_df, feature_cols)
        print(f"Pairs: {len(X):,}, Classes: {np.bincount(y)}")  #if y has 0s and 1s, bincount might print like [4000,3800] meaning 4000 negatives and 3800 positives
        self.model.fit(X, y)
        train_acc=self.model.score(X, y)
        print(f"Pairwise train accuracy: {train_acc:.4f}")
        print(f"Trained")
    
    def predict(self, test_df):
        predictions=[]
        for qid, group in test_df.groupby('qid'):
            group=group.reset_index(drop=True)
            X=group[self.feature_cols].values
            scores=self.model.decision_function(X)
            """
            Even though model was trained on pair differences, this scoring works because:
            if score(i) > score(j) then w·x_i > w·x_j, meaning model prefers i over j.
            Tiny example:
            w = [2, -1]
            doc features [3, 10] -> score = 2*3 + (-1)*10 = -4
            doc features [5, 6] -> score = 2*5 + (-1)*6 = 4
            Second doc ranks higher.
            """
            for i in range(len(group)):
                predictions.append({
                    'qid': qid,
                    'label': group.loc[i, 'label'],
                    'score': scores[i]
                })
        return pd.DataFrame(predictions)

print("Pairwise defined ")

Pairwise defined 


In [9]:
class PointwiseLogisticRanker:
    """
    Phase 5 baseline reproduced 
    Pointwise multinomial logistic regression (labels 0/1/2).
    """
    def __init__(self, random_state=36):
        self.model=LogisticRegression(
            max_iter=1000,
            random_state=random_state,
            solver='lbfgs',
            multi_class='multinomial'
        )
        self.feature_cols=None

    def fit(self, train_df, feature_cols):
        self.feature_cols=feature_cols
        X=train_df[feature_cols].values
        y=train_df['label'].values
        self.model.fit(X, y)
        print("Trained (pointwise multinomial)")

    def predict(self, test_df):
        #Predicting class probabilities, converting to expected relevance score E[label]
        X=test_df[self.feature_cols].values
        proba=self.model.predict_proba(X)

        #Aligning columns with class labels 
        classes=self.model.classes_
        exp_score=np.zeros(len(test_df), dtype=float)
        for i, c in enumerate(classes):
            exp_score+=c * proba[:, i]

        pred=test_df[['qid', 'label']].copy()
        pred['score']=exp_score
        return pred

print("Pointwise baseline defined")


Pointwise baseline defined


## 8. LightGBM Ranker

In [10]:
class LightGBMRanker:
    def __init__(self, random_state=36):
        self.model = lgb.LGBMRanker(
        objective='lambdarank', metric='ndcg',
        label_gain=[0,1,3], eval_at=[1,3,5,10],
        n_estimators=1000, learning_rate=0.05,
        num_leaves=31, max_depth=6,
        min_child_samples=20,
        subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=random_state, verbosity=-1
        )

        """ 
        100 trees max
        0.1 => how big each tree's contribution is
        depth=6 -> trees cannot grow too deep
        31 -> maximum leaves per tree
        verbosity=-1 -> suppress LightGBM warnings
        """
        self.feature_cols=None
    
    def fit(self, train_df, vali_df, feature_cols):
        self.feature_cols=feature_cols
        train_df=train_df.sort_values('qid').reset_index(drop=True)
        vali_df=vali_df.sort_values('qid').reset_index(drop=True)
        assert train_df['qid'].is_monotonic_increasing
        assert vali_df['qid'].is_monotonic_increasing

        X_train=train_df[feature_cols].values
        """ 
        X_train =
                    [
                        [0.2, 1.3, 4.1],
                        [0.8, 0.4, 2.2],
                        ...
                    ]

        """
        y_train=train_df['label'].values
        group_train=train_df.groupby('qid').size().values
        #this creates number of documents per query eg. [3,2] for 3 docs in qid 1, 2 docs in qid 2
        X_vali=vali_df[feature_cols].values
        y_vali=vali_df['label'].values
        group_vali=vali_df.groupby('qid').size().values
        print(f"Train: {len(group_train)} queries, Vali: {len(group_vali)} queries")
        self.model.fit(
            X_train, y_train, group=group_train,
            eval_set=[(X_vali, y_vali)], eval_group=[group_vali],
            callbacks=[lgb.early_stopping(10, verbose=False)]
        )
        print(f"Trained (best: {self.model.best_iteration_})")
    
    def predict(self, test_df):
        X=test_df[self.feature_cols].values
        scores=self.model.predict(X)
        pred=test_df[['qid', 'label']].copy()
        pred['score']=scores
        return pred

print("LightGBM defined")

LightGBM defined


In [11]:
print("\n"+"="*80)
print("PHASE 6 (FOLD RUNNER): GENERATING FOLD-AWARE ARTIFACTS FOR PHASE 10")
print("="*80)

FOLD_NUMS=[1, 2, 3]
MODELS=['pointwise','pairwise','lightgbm']
EXPECTED_CONFIGS=len(MODELS) * len(PIPELINES)  # 9

def validate_fold_outputs(fold_out, dataset='2007'):
    qm_ok=0
    pred_ok=0
    missing=[]
    bad_cols=[]

    for model_name in MODELS:
        for pipeline in PIPELINES:
            key=f"{model_name}_{pipeline}_{dataset}"
            qm_file=fold_out / f"{key}_query_metrics.csv"
            pred_file=fold_out / f"{key}_predictions.csv"

            if not qm_file.exists() or not pred_file.exists():
                missing.append(key)
                continue

            qm=pd.read_csv(qm_file)
            pred=pd.read_csv(pred_file)

            req_qm=['qid','num_docs','num_relevant_1','Failure@5_primary']
            req_pred=['qid','label','score']

            ok=True
            for c in req_qm:
                if c not in qm.columns:
                    ok=False
                    bad_cols.append((key, 'query_metrics', c))
            for c in req_pred:
                if c not in pred.columns:
                    ok=False
                    bad_cols.append((key, 'predictions', c))

            if ok:
                qm_ok+=1
                pred_ok+=1

    if len(missing)>0:
        print(f"Missing files: {len(missing)}")
        for m in missing:
            print(f" -{m}")

    if len(bad_cols)>0:
        print("Bad/missing columns:")
        for item in bad_cols:
            print(item)

    if qm_ok==EXPECTED_CONFIGS and pred_ok==EXPECTED_CONFIGS:
        print(f"{fold_out.name}: saved {qm_ok}/{EXPECTED_CONFIGS} configs (OK)")
        return True
    else:
        print(f"{fold_out.name}: saved {qm_ok}/{EXPECTED_CONFIGS} query_metrics, {pred_ok}/{EXPECTED_CONFIGS} predictions (NOT OK)")
        return False


for fold_num in FOLD_NUMS:
    fold_name=f"Fold{fold_num}"
    fold_out=PHASE6_FOLDS_OUTPUT / fold_name
    fold_out.mkdir(parents=True, exist_ok=True)

    print("\n"+"="*80)
    print(f"RUNNING MQ2007 {fold_name} (Phase 6 fold-aware)")
    print("="*80)

    #Fold-specific data load (MQ2007 only)
    train_2007_f, vali_2007_f, test_2007_f=load_letor_fold('MQ2007', fold_num)
    feature_cols_f=[c for c in train_2007_f.columns if c.startswith('f')]
    print(f"\nFeatures: {len(feature_cols_f)}")

    #Fresh models per fold (same params)
    models_config_f={
        'pointwise': PointwiseLogisticRanker(RANDOM_SEED),
        'pairwise': PairwiseLogisticRanker(RANDOM_SEED, MAX_PAIRS_PER_QUERY),
        'lightgbm': LightGBMRanker(RANDOM_SEED)
    }

    all_predictions_f={}
    all_query_metrics_f={}

    for model_name, model in models_config_f.items():
        print(f"\n{'='*80}\nMODEL: {model_name.upper()} ({fold_name})\n{'='*80}")

        for pipeline in PIPELINES:
            print(f"\n{'-'*80}\nPipeline: {pipeline}\n{'-'*80}")

            train_norm, test_norm, scaler=apply_normalization(train_2007_f, test_2007_f, pipeline, feature_cols_f)
            _, vali_norm, _=apply_normalization(train_2007_f, vali_2007_f, pipeline, feature_cols_f, scaler)

            if model_name=='lightgbm':
                model.fit(train_norm, vali_norm, feature_cols_f)
            else:
                model.fit(train_norm, feature_cols_f)

            print(f"\nPredicting MQ2007 ({fold_name})...")
            pred_2007=model.predict(test_norm)
            metrics_2007=evaluate_query_level(pred_2007, K_VALUES)

            key=f"{model_name}_{pipeline}_2007"
            all_predictions_f[key]=pred_2007
            all_query_metrics_f[key]=metrics_2007

            agg=metrics_2007[['P@5_primary','NDCG@5','Failure@5_primary']].mean()
            print(f"\nMQ2007 {fold_name}: P@5={agg['P@5_primary']:.4f}, NDCG@5={agg['NDCG@5']:.4f}, Fail@5={agg['Failure@5_primary']:.4f}")

    print("\n"+"="*80)
    print(f"SAVING PHASE 10 ARTIFACTS ({fold_name})")
    print("="*80)

    for model_name in MODELS:
        for pipeline in PIPELINES:
            key=f"{model_name}_{pipeline}_2007"
            all_predictions_f[key].to_csv(fold_out / f"{key}_predictions.csv", index=False)
            all_query_metrics_f[key].to_csv(fold_out / f"{key}_query_metrics.csv", index=False)

    print("\n"+"="*80)
    print(f"VALIDATION ({fold_name})")
    print("="*80)

    validate_fold_outputs(fold_out, dataset='2007')


print("\n"+"="*80)
print("FOLD-AWARE OUTPUT TREE (phase6_models_folds)")
print("="*80)

for fold_num in FOLD_NUMS:
    fold_name=f"Fold{fold_num}"
    fold_out=PHASE6_FOLDS_OUTPUT / fold_name
    print(f"\n{fold_name}:")
    for f in sorted(fold_out.glob("*")):
        print(f" -{f.name}")

print("\nNOTE:")
print("Phase 10 should set PHASE6_OUTPUT to:")
print(f"{PHASE6_FOLDS_OUTPUT}")


PHASE 6 (FOLD RUNNER): GENERATING FOLD-AWARE ARTIFACTS FOR PHASE 10

RUNNING MQ2007 Fold1 (Phase 6 fold-aware)
MQ2007 train:  42158 docs, 1017 queries
MQ2007 vali :  13813 docs,  339 queries
MQ2007 test :  13652 docs,  336 queries

Features: 41

MODEL: POINTWISE (Fold1)

--------------------------------------------------------------------------------
Pipeline: raw
--------------------------------------------------------------------------------
Trained (pointwise multinomial)

Predicting MQ2007 (Fold1)...

MQ2007 Fold1: P@5=0.4220, NDCG@5=0.4283, Fail@5=0.2679

--------------------------------------------------------------------------------
Pipeline: global
--------------------------------------------------------------------------------
Trained (pointwise multinomial)

Predicting MQ2007 (Fold1)...

MQ2007 Fold1: P@5=0.4173, NDCG@5=0.4249, Fail@5=0.2738

--------------------------------------------------------------------------------
Pipeline: per_query
---------------------------------